# 05 — Neural Networks and the Modular Layer API

**CS231n Lecture 4–5.** Notebook 02 ended with the horse that had two heads: a
linear classifier gets exactly one template per class, so any class with more
than one appearance gets averaged into mush.

The fix is one line of algebra:

$$ \text{linear:}\quad s = W x \qquad\qquad \text{2-layer:}\quad s = W_2 \max(0, W_1 x) $$

$W_1$ learns a bank of templates — left-facing horse, right-facing horse, red
car, blue car — and $W_2$ learns to **combine** them. The `max(0, ·)` is what
makes this more than a product of two matrices.

The other half of this notebook matters just as much: you stop writing
monolithic loss functions and start writing **layers**.

| You write | Contract |
|---|---|
| `affine_forward` / `affine_backward` | the workhorse |
| `relu_forward` / `relu_backward` | the nonlinearity |
| `affine_relu_forward` / `_backward` | composing two layers into one |
| `softmax_loss`, `svm_loss` | losses as layers |
| `TwoLayerNet.loss` | assemble the whole thing |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (get_cifar10, CLASSES, rel_error, grade, show_solution,
                          use, show_images, plot_series, eval_numerical_gradient,
                          eval_numerical_gradient_array, setup_plots, SERIES)

setup_plots()
np.random.seed(0)

data = get_cifar10(num_train=5000, num_val=500, num_test=500,
                   flatten=True, subtract_mean=True)
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']

# Mean subtraction alone is not enough here. CIFAR pixels still have a standard
# deviation around 64, and an affine layer's initial activations scale with it:
# with W ~ 1e-2 and 3072 inputs, the hidden units come out with std ~35 and the
# net is unstable from the first step. Divide by the training std so every
# feature is roughly unit scale.
pixel_std = data['X_train'].std()
X_train = data['X_train'] / pixel_std
X_val   = data['X_val']   / pixel_std     # train statistics, never val/test's
X_test  = data['X_test']  / pixel_std

print("X_train", X_train.shape)
print("std before %.2f  ->  after %.3f" % (pixel_std, X_train.std()))

## 1. Why the nonlinearity is not optional

Without it:

$$ W_2 (W_1 x) = (W_2 W_1) x = W_{\text{eff}}\, x $$

The product of two matrices is a matrix. A hundred stacked linear layers is
still a linear classifier with extra steps and worse conditioning. **The
nonlinearity is the only reason depth buys you anything.**

In [ ]:
# Demonstrate it: stacked linear layers collapse to one matrix.
rng = np.random.RandomState(0)
W1, W2, W3 = rng.randn(6, 5), rng.randn(5, 4), rng.randn(4, 3)
xin = rng.randn(2, 6)

stacked = xin.dot(W1).dot(W2).dot(W3)
collapsed = xin.dot(W1.dot(W2).dot(W3))
print("3 linear layers vs 1 equivalent matrix: rel error %.2e"
      % rel_error(stacked, collapsed))

with_relu = np.maximum(0, np.maximum(0, xin.dot(W1)).dot(W2)).dot(W3)
print("with ReLU in between, same comparison:  rel error %.2e  <- no longer equal"
      % rel_error(with_relu, collapsed))

## 2. The layer API

Every layer from here to the end of the series obeys this contract:

```python
out, cache = layer_forward(x, *params)
dx, *dparams = layer_backward(dout, cache)
```

`cache` is whatever backward needs and nothing more. Forward and backward share
**no other state**. That is what makes layers composable — you can stack them in
any order and the gradients still work, because each one only ever talks to its
immediate neighbours.

This is exactly the locality principle from notebook 04, turned into a code
convention.

### TODO 1 — the affine layer

Two details that are easy to miss:

- **`x` may not be 2D.** A convnet hands you `(N, C, H, W)`. Flatten to
  `(N, D)` with `D = C*H*W` for the multiply, and on the backward pass reshape
  `dx` back to `x`'s **original** shape.
- **`db` is a sum, not a copy.** `b` is broadcast across all `N` rows on the
  forward pass. Broadcasting is a copy, a copy is a branch, and branches add
  their gradients (notebook 04, section 6). So `db = dout.sum(axis=0)`.

In [ ]:
def affine_forward(x, w, b):
    """x (N, d1, ..., dk), w (D, M), b (M,)  ->  (out (N, M), cache)"""
    pass  # TODO

def affine_backward(dout, cache):
    """dout (N, M)  ->  (dx like x, dw like w, db like b)"""
    pass  # TODO

In [ ]:
grade('nb05', 'affine_forward', affine_forward, verbose=False)
grade('nb05', 'affine_backward', affine_backward, verbose=False)

In [ ]:
af = use('nb05', 'affine_forward', affine_forward)
ab = use('nb05', 'affine_backward', affine_backward)

# note the 3D input -- this is the reshape case
x = rng.randn(4, 2, 3)
w, b = rng.randn(6, 5), rng.randn(5)
out, cache = af(x, w, b)
dout = rng.randn(*out.shape)
dx, dw, db = ab(dout, cache)

print("x %s -> out %s;  dx %s (must match x)" % (x.shape, out.shape, dx.shape))
print("dx rel error %.2e" % rel_error(dx, eval_numerical_gradient_array(
    lambda t: af(t, w, b)[0], x.copy(), dout)))
print("dw rel error %.2e" % rel_error(dw, eval_numerical_gradient_array(
    lambda t: af(x, t, b)[0], w.copy(), dout)))
print("db rel error %.2e" % rel_error(db, eval_numerical_gradient_array(
    lambda t: af(x, w, t)[0], b.copy(), dout)))

### TODO 2 — ReLU

$$ \text{ReLU}(x) = \max(0, x) $$

The max gate from notebook 04, with one input pinned to zero: gradient passes
where the input was positive, is blocked where it was not.

At exactly `x = 0` the derivative does not exist. Everyone uses `0` (i.e.
`x > 0`, not `x >= 0`) and it never matters, because exact zeros essentially
never occur in float arithmetic.

In [ ]:
def relu_forward(x):
    """-> (out, cache)"""
    pass  # TODO

def relu_backward(dout, cache):
    """-> dx"""
    pass  # TODO

In [ ]:
grade('nb05', 'relu_forward', relu_forward, verbose=False)
grade('nb05', 'relu_backward', relu_backward, verbose=False)

### TODO 3 — the sandwich layer

`affine` then `relu` is so common it gets its own function. This is the whole
point of the API: you compose forwards in order, and backwards **in reverse
order**, and nothing else changes.

In [ ]:
def affine_relu_forward(x, w, b):
    """affine then relu.  -> (out, cache) where cache holds both sub-caches."""
    pass  # TODO

def affine_relu_backward(dout, cache):
    """-> (dx, dw, db).  Backward through relu FIRST, then affine."""
    pass  # TODO

In [ ]:
grade('nb05', 'affine_relu_forward', affine_relu_forward, verbose=False)
grade('nb05', 'affine_relu_backward', affine_relu_backward, verbose=False)

### TODO 4 — losses as layers

Same idea, at the end of the network: a loss takes scores and labels and returns
`(loss, dx)`. `dx` is the gradient that starts the whole backward pass.

For softmax, work in **log space**:

```
shifted   = x - x.max(axis=1, keepdims=True)
log_probs = shifted - log(sum(exp(shifted), axis=1, keepdims=True))
loss      = -mean(log_probs[arange(N), y])
```

Computing `log(p)` as `log(exp(shifted)/sum)` loses precision for small
probabilities; subtracting the log-sum-exp does not. The gradient is still
`p - y_onehot`, and `p = exp(log_probs)`.

In [ ]:
def softmax_loss(x, y):
    """x (N, C) scores, y (N,) labels  ->  (loss, dx), averaged over N."""
    pass  # TODO

def svm_loss(x, y):
    """Hinge loss with delta=1 as a layer  ->  (loss, dx), averaged over N."""
    pass  # TODO

In [ ]:
grade('nb05', 'softmax_loss', softmax_loss, verbose=False)
grade('nb05', 'svm_loss', svm_loss, verbose=False)

In [ ]:
sl = use('nb05', 'softmax_loss', softmax_loss)
vl = use('nb05', 'svm_loss', svm_loss)

xs = rng.randn(10, 5) * 2.0
ys = rng.randint(0, 5, 10)
for nm, fn in [('softmax', sl), ('svm', vl)]:
    l, dxl = fn(xs, ys)
    num = eval_numerical_gradient(lambda t: fn(t, ys)[0], xs.copy())
    print("%-8s loss %.4f   dx rel error %.2e" % (nm, l, rel_error(dxl, num)))

## 3. TODO 5 — assemble the network

Now the payoff. `TwoLayerNet.loss` does the whole forward and backward by
calling the layers you just wrote. No new maths — the maths is done.

Structure: `affine - relu - affine - softmax`.

Regularisation convention **changes here** and it matters: this notebook uses
$\frac{1}{2}\lambda \sum W^2$ (note the one-half), so the gradient is
$\lambda W$ with no factor of 2. Notebooks 02–03 used $\lambda \sum W^2$ and
$2\lambda W$. Both are fine; mixing them inside one file is not. From here on,
the one-half convention.

In [ ]:
from solutions.nb05 import (affine_forward as AF, affine_backward as AB,
                            affine_relu_forward as ARF,
                            affine_relu_backward as ARB)

class TwoLayerNet(object):
    """affine - relu - affine - softmax"""

    def __init__(self, input_dim=3072, hidden_dim=100, num_classes=10,
                 weight_scale=1e-3, reg=0.0, seed=None):
        rng_ = np.random.RandomState(seed)
        self.reg = reg
        self.params = {
            'W1': weight_scale * rng_.randn(input_dim, hidden_dim),
            'b1': np.zeros(hidden_dim),
            'W2': weight_scale * rng_.randn(hidden_dim, num_classes),
            'b2': np.zeros(num_classes),
        }

    def loss(self, X, y=None):
        W1, b1 = self.params['W1'], self.params['b1']
        W2, b2 = self.params['W2'], self.params['b2']

        # TODO forward:
        #   h, cache1      = affine_relu_forward(X, W1, b1)
        #   scores, cache2 = affine_forward(h, W2, b2)
        scores, cache1, cache2 = None, None, None

        if y is None:
            return scores

        # TODO:
        #   loss, dscores = softmax_loss(scores, y)
        #   loss += 0.5 * reg * (sum(W1^2) + sum(W2^2))
        #   backward through cache2 then cache1, and add reg*W to each dW
        loss, grads = 0.0, {}
        return loss, grads

### Sanity checks before you train anything

Three of them, in this order. Every one has caught a real bug for someone.

In [ ]:
use_ref = False
try:
    _net = TwoLayerNet(input_dim=8, hidden_dim=6, num_classes=4,
                       weight_scale=0.1, reg=0.0, seed=1)
    _l, _g = _net.loss(np.random.randn(5, 8), np.random.randint(0, 4, 5))
    use_ref = (_l == 0.0) or not _g
except Exception:
    use_ref = True

if use_ref:
    from solutions.nb05 import TwoLayerNet as NetClass
    print("(using the reference TwoLayerNet -- yours is not filled in yet)")
else:
    NetClass = TwoLayerNet

In [ ]:
# CHECK 1 -- initial loss. Small weights -> near-uniform scores -> -log(1/C).
net = NetClass(input_dim=20, hidden_dim=10, num_classes=10,
               weight_scale=1e-3, reg=0.0, seed=0)
Xq = np.random.RandomState(5).randn(50, 20)
yq = np.random.RandomState(6).randint(0, 10, 50)
print("initial loss %.4f   expected %.4f" % (net.loss(Xq, yq)[0], np.log(10)))

# Turning regularisation on must RAISE the loss, monotonically.
# The weights are tiny at init, so the penalty is small -- print enough digits.
for r in [0.0, 1.0, 100.0, 1000.0]:
    net.reg = r
    print("  reg %7.1f  ->  loss %.6f" % (r, net.loss(Xq, yq)[0]))
net.reg = 0.0

In [ ]:
# CHECK 2 -- gradient check every parameter, on a tiny net in float64.
net = NetClass(input_dim=8, hidden_dim=6, num_classes=4,
               weight_scale=0.1, reg=0.3, seed=1)
Xn = np.random.RandomState(2).randn(7, 8)
yn = np.random.RandomState(3).randint(0, 4, 7)
loss, grads = net.loss(Xn, yn)
print("loss %.4f" % loss)
for p in sorted(net.params):
    orig = net.params[p].copy()
    def f(t, _p=p, _o=orig):
        net.params[_p] = t
        v = net.loss(Xn, yn)[0]
        net.params[_p] = _o
        return v
    num = eval_numerical_gradient(f, orig.copy())
    print("  %-3s rel error %.3e" % (p, rel_error(num, grads[p])))

> **Reading a gradient check.** Below `1e-7` is clean. Around `1e-5` you should
> be suspicious unless there are kinks (ReLU has them). Above `1e-3` there is a
> bug. Do the check on a **small** net in float64 — a big net has more chances
> to hit a kink, and float32 has no precision to spare.

In [ ]:
# CHECK 3 -- overfit 50 examples. A correct net must reach ~100% train accuracy
# on a tiny subset. If it cannot memorise 50 images, it will never learn 50,000.
from solutions.nb05 import train_net

tiny = NetClass(input_dim=3072, hidden_dim=100, num_classes=10,
                weight_scale=1e-2, reg=0.0, seed=0)
h = train_net(tiny, X_train[:50], y_train[:50], X_val[:50], y_val[:50],
              learning_rate=1e-2, num_epochs=30, batch_size=25,
              lr_decay=1.0, verbose=False)
print("train accuracy on 50 examples: %.3f  (want 1.000)" % h['train_acc'][-1])
print("final loss %.5f  (want ~0)" % h['loss'][-1])

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(h['loss'], color=SERIES[0])
ax.set_title("Overfitting 50 examples on purpose")
ax.set_xlabel("iteration"); ax.set_ylabel("loss")
ax.set_axisbelow(True)
plt.show()

**This is the single best debugging check in deep learning.** Turn off
regularisation, take 20–50 examples, and train until the loss is near zero. It
isolates the model and the optimiser from everything else: if you cannot
overfit a handful of examples, the problem is a bug, not your hyperparameters,
and no amount of tuning will save you.

## 4. Train it for real

In [ ]:
model = NetClass(input_dim=3072, hidden_dim=200, num_classes=10,
                 weight_scale=1e-2, reg=0.05, seed=0)
t0 = time.time()
hist = train_net(model, X_train, y_train, X_val, y_val,
                 learning_rate=1e-2, momentum=0.9, num_epochs=12,
                 batch_size=200, lr_decay=0.95, verbose=True)
print("\n%.1f s" % (time.time() - t0))

scores = model.loss(X_test)
print("TEST accuracy %.3f" % np.mean(np.argmax(scores, axis=1) == y_test))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5))
axes[0].plot(hist['loss'], color=SERIES[0], linewidth=1.0, alpha=0.6)
k = 20
axes[0].plot(np.arange(k - 1, len(hist['loss'])),
             np.convolve(hist['loss'], np.ones(k) / k, mode='valid'),
             color=SERIES[0])
axes[0].set_title("Training loss")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("loss")
axes[0].set_axisbelow(True)

plot_series(axes[1], np.arange(1, len(hist['train_acc']) + 1),
            [('train', hist['train_acc']), ('val', hist['val_acc'])],
            "Accuracy", "epoch", "accuracy")
plt.show()

print("final train %.3f  val %.3f  gap %.3f"
      % (hist['train_acc'][-1], hist['val_acc'][-1],
         hist['train_acc'][-1] - hist['val_acc'][-1]))

### Reading the accuracy curves

The gap between the two lines is the diagnosis:

| Pattern | Meaning | What to do |
|---|---|---|
| both low, still rising | underfitting, or not trained long enough | bigger model, train longer, higher LR |
| large and widening gap | overfitting | more regularisation, more data, smaller model |
| small gap, both plateaued | capacity-limited | bigger model |
| val **above** train | almost always a bug in how you measure | check your splits |

That last row deserves a note: it can happen legitimately when train accuracy is
measured during training (with dropout on) and val without. Absent that, suspect
leakage between splits.

In [ ]:
# What the first layer learned. Compare to the linear templates in notebook 02.
W1 = model.params['W1']
n_show = 64
grid = W1[:, :n_show].T.reshape(n_show, 32, 32, 3)
show_images(grid, ncols=16, title="First-layer weights (64 of %d hidden units)"
            % W1.shape[1], scale=1.0)
plt.show()

### Be honest about what you are seeing

These mostly look like **noise**, with some diffuse colour blobs. If you were
expecting the tidy oriented edges and Gabor-like stripes from the textbook
pictures, they are not here — and the reason is worth more than the picture
would have been.

What *has* changed, and it is the thing that matters: there are now 200
different patterns instead of 10 class templates. The second layer is free to
combine them, so the one-prototype-per-class ceiling from notebook 02 is gone.
A left-facing horse and a right-facing horse can excite different hidden units
and $W_2$ can add both into the horse score.

Why they look like noise:

- **A fully-connected layer has no reason to prefer local structure.** Every
  hidden unit is wired to all 3072 pixels with independent weights. Nothing in
  the architecture says neighbouring pixels are related, so nothing pushes a
  unit toward a compact oriented edge rather than a diffuse global pattern.
- **Nothing ties a feature at one position to the same feature elsewhere.** A
  unit that learns a corner in the top-left is useless for the same corner in
  the bottom-right, so the limited training signal is spread thin.
- 5,000 images and 12 epochs is simply not enough to constrain 614,400 weights.

Those first two are not data problems, they are **architecture** problems, and
no amount of extra training fixes them. Local connectivity and weight sharing
fix them — which is exactly what convolution is. Notebook 06.

> A caveat on comparisons: the conv filters you will see in notebook 09 are
> 3×3, so each one has nine pixels to show you. Do not read their tidiness as
> "convolution learns cleaner features than this" — a 9-pixel kernel and a
> 3072-pixel weight vector are not comparable pictures. The honest claim is
> about *constraints*, not aesthetics: convolution restricts each unit to a
> small neighbourhood and reuses it everywhere, so there are vastly fewer
> parameters to pin down and every one of them gets gradient from every image
> position.

## 5. Recap

1. Why is `W2 @ W1 @ x` not a two-layer network?
2. State the layer API contract in one line. What does `cache` hold?
3. Why is `db` a sum over the batch rather than a copy of `dout`?
4. Why must `affine_backward` reshape `dx` rather than return it flat?
5. Why compute log-probabilities as `shifted - logsumexp` instead of
   `log(softmax)`?
6. Name the three pre-training sanity checks, in order, and what each rules out.
7. Your gradient check gives `4e-3` on `W1` and `1e-9` on everything else.
   Where is the bug?
8. Train 0.62, val 0.38. What is happening and what are two fixes?
9. Why do first-layer weights look like edges here, but like blurry class
   templates in notebook 02?

In [ ]:
# show_solution('nb05', 'affine_backward')
# show_solution('nb05', 'softmax_loss')
# show_solution('nb05', 'TwoLayerNet')